# 16 — MCP concepts: a standard boundary for tools and context

The Model Context Protocol (MCP) is an open standard, introduced by Anthropic in November 2024, for connecting a host application to servers that expose capabilities. It does not run your agent; it standardizes the boundary the agent reaches through.

**Roles.** A *host* (chat app, IDE, agent harness) owns the model and the loop. It runs one *client* per connection. A *server* exposes capabilities. Because the wire contract is fixed, any host can use any server without bespoke glue.

**Primitives.** Servers expose three things, distinguished by who decides to use them:

| Primitive | Controlled by | Example |
| --- | --- | --- |
| Tools | the model | `search_flights(origin, destination)` |
| Resources | the application | a file, schema, or log offered as context |
| Prompts | the user | a reusable slash-command template |

Clients can also offer *sampling* (the server asks the host's model for a completion), *elicitation* (ask the user for input), and *roots* (filesystem boundaries) — the flow can point both ways.

**Wire format.** Everything is JSON-RPC 2.0 over a transport — stdio for local subprocess servers, streamable HTTP for remote ones. A session starts with an `initialize` handshake that negotiates protocol version and capabilities, then the client discovers and invokes primitives (`tools/list`, `tools/call`, `resources/read`, ...).

**Where this fits the lab.** In notebook 01 the tool contract was a Python dict you wired by hand. MCP standardizes exactly that boundary: the loop, budgets, and verification stay yours (notebooks 01–05); the tool catalog becomes discoverable and portable. The cells below use a deterministic in-memory teaching double from `lab_core.mcp_toy` so every message is visible; notebook 17 repeats the exercise with the official SDK over a real transport.

Primary sources: [modelcontextprotocol.io](https://modelcontextprotocol.io/), the [specification](https://modelcontextprotocol.io/specification/2025-06-18), and the [announcement](https://www.anthropic.com/news/model-context-protocol).

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'notebooks'))
from lab_core.mcp_toy import ToyMCPServer, ToyMCPClient

def get_forecast(args):
    forecasts = {'oslo': 'Snow, -3C', 'lisbon': 'Sunny, 21C'}
    city = str(args['city']).lower()
    if city not in forecasts:
        raise ValueError(f'no forecast for {city}')
    return forecasts[city]

server = ToyMCPServer('weather-lab')
server.add_tool(
    'get_forecast',
    'Deterministic forecast for a known city.',
    {'type': 'object', 'properties': {'city': {'type': 'string'}}, 'required': ['city']},
    get_forecast,
)
server.add_resource('lab://cities', 'Supported cities', 'oslo\nlisbon')
server.add_prompt('brief', 'Weather briefing request', 'Give a one-line weather brief for {city}.')

client = ToyMCPClient(server)
capabilities = client.connect()
for entry in client.transcript:  # the raw initialize handshake, message by message
    print(json.dumps(entry, indent=2))

In [ ]:
tools = client.list_tools()
print('discovered:', [(t['name'], t['description']) for t in tools])

print(client.call_tool('get_forecast', {'city': 'Oslo'}))
print(client.call_tool('get_forecast', {'city': 'Atlantis'}))  # isError result, not a crash
print(client.read_resource('lab://cities')['contents'][0]['text'])
print(client.get_prompt('brief', {'city': 'Lisbon'})['messages'][0]['content']['text'])

In [ ]:
# MCP as the tool boundary of the notebook-01 loop: the policy and budget stay
# ours; the tool catalog and invocation go through the protocol.
from lab_core.patterns import run_tool_loop

mcp_tools = {
    t['name']: (lambda args, name=t['name']:
                client.call_tool(name, args)['content'][0]['text'])
    for t in client.list_tools()
}

def policy(goal, trace):
    if not trace:
        return {'tool': 'get_forecast', 'input': {'city': 'Oslo'}}
    return {'final': f"{goal}: {trace[-1]['observation']}"}

result = run_tool_loop('Oslo forecast', policy, mcp_tools, max_steps=3)
print(result.answer)
print('JSON-RPC messages exchanged this session:', len(client.transcript))

**Trade-off.** MCP buys portability and discovery at the cost of a subprocess or network hop and a trust decision: a server's tool descriptions enter your prompt, so treat third-party servers as untrusted input and keep the approval gates from notebook 05 at consequential boundaries.